## Microsoft Agent Framework (MAF): Ramp-Up Training Materials

To get the latest version of MAF Python package, use:

``` bash
pip install --upgrade agent-framework --pre
pip install --upgrade azure-ai-projects --pre
pip install --upgrade azure-monitor-opentelemetry-exporter
```

## 📒 Notebook 3: Observability

### 🪜 Step 1: Configure environment

In [1]:
# Suppress warnings from agent_framework_azure module
import warnings
warnings.filterwarnings(
    action = "ignore",
    category = UserWarning,
    module = "agent_framework_azure"
)

In [2]:
# Import required packages
import os
import asyncio
from agent_framework import agent_middleware, ChatMessage, AgentRunResponse
from agent_framework.azure import AzureAIClient
from azure.identity.aio import DefaultAzureCredential

In [3]:
# Fix: Disable Accept-Encoding to prevent compressed responses
import azure.core.pipeline.policies as policies

# Store original
_original_on_request = policies.HeadersPolicy.on_request

def patched_on_request(self, request):
    """Remove Accept-Encoding header to prevent compressed responses."""
    _original_on_request(self, request)
    # Tell server we don't accept compressed responses
    request.http_request.headers['Accept-Encoding'] = 'identity'

policies.HeadersPolicy.on_request = patched_on_request
print("Applied Accept-Encoding fix to disable compression")

Applied Accept-Encoding fix to disable compression


In [4]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Create AI agent

In [5]:
# Define AI client
ai_client = AzureAIClient(
    agent_name = "MAFSDK-Agentv2-FinAdvisor",
    project_endpoint = PROJECT_ENDPOINT,
    model_deployment_name = MODEL_DEPLOYMENT,
    async_credential = DefaultAzureCredential()
)

print(f"Created AI client: {ai_client.agent_name}")

Created AI client: MAFSDK-Agentv2-FinAdvisor


In [6]:
# Create Fin Advisor agent
agent = ai_client.create_agent(
    name = "financial-advisor-agent",
    instructions = "You are a helpful financial advisor.",
)

print(f"Created AI agent: {agent.name}")

Created AI agent: financial-advisor-agent


### 🪜 Step 3: Enable observability

Your Azure Ai Foundry requires connection to Application Insights to be setup in advance.

In [7]:
# Setup observability
await ai_client.setup_azure_ai_observability()

### 🪜 Step 4: Run the agent

In [8]:
# Test chat agent with a prompt
prompt = "Please, explain the top 3 differences in the tax submission process between UK and US."
print(f"User:\n{prompt}\n")

response = await agent.run(prompt)
print(f"Agent:\n{response.text}")

User:
Please, explain the top 3 differences in the tax submission process between UK and US.

Agent:
Certainly! Here are the top 3 differences in the tax submission process between the UK and the US:

1. **Filing System: Self-Assessment vs. Withholding and Annual Filing**
   - **UK:** Most employees have income tax automatically deducted through the Pay As You Earn (PAYE) system by their employer. However, individuals who are self-employed or have other income sources must file a Self-Assessment tax return annually.
   - **US:** The US tax system relies heavily on annual tax returns where all taxpayers must file a federal tax return (Form 1040) each year. Employers withhold income tax, Social Security, and Medicare from paychecks, but taxpayers still need to reconcile their tax obligations through filing.

2. **Tax Year and Deadlines**
   - **UK:** The tax year runs from April 6 to April 5 the following year. The deadline for online Self-Assessment filing is January 31 following the en

### 🪜 Step 5: Housekeeping

In [9]:
# Delete Azure AI client
await ai_client.close()

print("AI client closed successfully.")

AI client closed successfully.
